In [1]:
import os
import numpy as np
import pandas as pd
from ogcore.utils import safe_read_pickle
from ogcore.output_tables import dynamic_revenue_decomposition

In [2]:
CUR_DIR = './'
base_dir = os.path.join(CUR_DIR, 'Current_Law', "OUTPUT")
reform_dir = os.path.join(CUR_DIR, 'TCJA_Ext', "OUTPUT")

base_tpi = safe_read_pickle(os.path.join(base_dir, "TPI", "TPI_vars.pkl"))
base_params = safe_read_pickle(os.path.join(base_dir, "model_params.pkl"))
base_ss = safe_read_pickle(os.path.join(base_dir, "SS", "SS_vars.pkl"))
reform_tpi = safe_read_pickle(os.path.join(reform_dir, "TPI", "TPI_vars.pkl"))
reform_params = safe_read_pickle(os.path.join(reform_dir, "model_params.pkl"))
reform_ss = safe_read_pickle(os.path.join(reform_dir, "SS", "SS_vars.pkl"))

In [3]:
df = dynamic_revenue_decomposition(base_params, base_tpi, base_ss, reform_params, reform_tpi, reform_ss, start_year=2025, num_years=10, full_break_out=True)
df

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034,SS
0,IIT: Pct Change due to tax rates,-5.44,-5.44,-5.44,-5.44,-5.44,-5.44,-5.44,-5.44,-5.44,-5.44,-5.44,-5.44
1,IIT: Pct Change due to behavior,1.24,1.22,1.22,1.21,1.20,1.19,1.19,1.19,1.19,1.18,1.20,1.20
2,IIT: Pct Change due to macro,-0.03,-0.04,-0.04,-0.06,-0.07,-0.08,-0.10,-0.11,-0.13,-0.15,-0.08,0.17
3,IIT: Overall Pct Change in taxes,-4.29,-4.32,-4.33,-4.35,-4.37,-4.39,-4.41,-4.43,-4.44,-4.46,-4.38,-4.14
4,CIT: Pct Change due to tax rates,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
5,CIT: Pct Change due to behavior,0.93,0.97,1.01,1.02,1.03,1.04,1.03,1.02,1.01,0.99,1.01,1.91
6,CIT: Pct Change due to macro,0.49,0.38,0.28,0.21,0.16,0.12,0.08,0.06,0.05,0.04,0.19,-0.92
7,CIT: Overall Pct Change in taxes,1.42,1.35,1.29,1.24,1.19,1.15,1.12,1.09,1.06,1.03,1.19,0.97
8,All: Pct Change due to tax rates,-5.13,-5.13,-5.13,-5.13,-5.13,-5.13,-5.13,-5.13,-5.13,-5.13,-5.13,-5.12
9,All: Pct Change due to behavior,1.23,1.21,1.20,1.20,1.19,1.18,1.18,1.18,1.17,1.17,1.19,1.24


In [4]:
# Now apply these percentage changes to the baseline revenue
# Take CBO baseline (to include not just IIT)
# Taken from CBO June 2024 Budgdet Outlook, 2026-2034
base_revenue = np.array([5.038, 5.394, 5.756, 5.944, 6.133, 6.354, 6.661, 6.899, 7.176, 7.459])
# Above is just over all revenue so only apply to those rows
df_levels = df.loc[8:, df.columns[:-2]]
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 10), (df_levels.shape[0], 1)) / 100
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
8,Rev Change Due to Tax Rates,-0.26,-0.28,-0.30,-0.30,-0.31,-0.33,-0.34,-0.35,-0.37,-0.38,-3.22
9,Rev Change Due to Behavior,0.06,0.07,0.07,0.07,0.07,0.08,0.08,0.08,0.08,0.09,0.75
10,Rev Change Due to Macro,0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.01,-0.01,-0.01,-0.01,-0.04
11,Total Revenue Change,-0.20,-0.22,-0.23,-0.24,-0.25,-0.26,-0.27,-0.28,-0.30,-0.31,-2.55


In [5]:
result_df_static = pd.read_csv('../../Tax-Calculator-thru74/tax_brain_result_wo_behresp.csv', index_col = 0)

In [6]:
# Or we can use the Tax-Calc baseline for a direct comparison
base_revenue = result_df_static.loc["Base", ['2025', '2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values
# Above is just over all revenue so only apply to those rows
df_levels = df.loc[0:3, df.columns[:-2]]
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 10), (df_levels.shape[0], 1)) / 100
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
0,Rev Change Due to Tax Rates,-0.22,-0.25,-0.26,-0.27,-0.28,-0.29,-0.30,-0.32,-0.33,-0.34,-2.86
1,Rev Change Due to Behavior,0.05,0.06,0.06,0.06,0.06,0.06,0.07,0.07,0.07,0.07,0.63
2,Rev Change Due to Macro,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.01,-0.01,-0.01,-0.01,-0.04
3,Total Revenue Change,-0.18,-0.20,-0.21,-0.22,-0.23,-0.24,-0.25,-0.26,-0.27,-0.28,-2.31


In [7]:
# jason's get-around

df_levels = df.loc[0:3, df.columns[:-2]]
tc_diff = result_df_static.loc["Difference", ['2025', '2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values
tc_reform = result_df_static.loc["Reform", ['2025', '2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values
df_levels.loc[0, df_levels.columns[1:]] = tc_diff
df_levels.loc[1, df_levels.columns[1:]] = (df.loc[1, df_levels.columns[1:]] / 100) * tc_reform
df_levels.loc[2, df_levels.columns[1:]] = (df.loc[2, df_levels.columns[1:]] / 100) * df_levels.loc[1, df_levels.columns[1:]]
df_levels.loc[3, df_levels.columns[1:]] = df_levels.loc[0, df_levels.columns[1:]] + df_levels.loc[1, df_levels.columns[1:]] + df_levels.loc[2, df_levels.columns[1:]]
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels = 

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
0,Rev Change Due to Tax Rates,0.00,-0.29,-0.30,-0.31,-0.32,-0.33,-0.34,-0.35,-0.36,-0.36,-2.95
1,Rev Change Due to Behavior,0.05,0.05,0.05,0.06,0.06,0.06,0.06,0.06,0.07,0.07,0.60
2,Rev Change Due to Macro,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00
3,Total Revenue Change,0.05,-0.24,-0.25,-0.25,-0.26,-0.27,-0.27,-0.28,-0.29,-0.29,-2.35


In [13]:
df_levels.to_csv('og_usa_result_w_tcja.csv')